# RCT audit, estimands, and uplift feature provenance

**Research question:** What does randomized assignment causally change, and is heterogeneous targeting credible?

This notebook is the review interface. Reusable transformations remain in `src/`, `scripts/`, and `sql/`; the notebook reads their committed evidence rather than creating a second implementation. Raw source data is intentionally not embedded.

## Hypotheses declared before interpretation

- Randomized arms should be balanced on f0-f11.
- Intent-to-treat is the primary estimand; exposure comparisons are observational.
- A targeting policy must validate uplift rank on untouched holdout data.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
print(f'Repository: {ROOT.name}')


Repository: causal-experimentation-lab


## Feature provenance

Every row states its raw source, transformation, decision-time cutoff, fit scope, missing-value treatment, and known leakage or proxy risk.

In [2]:
features = pd.read_csv(ROOT / 'reports' / 'feature_provenance.csv')
display(features)
display(features.groupby(['feature_family', 'model_role']).size().rename('features').reset_index())


,feature_name,feature_family,raw_source,grain,transformation,observation_cutoff,available_at_decision,fit_partition,missing_treatment,model_role,exclusion_reason,leakage_risk,proxy_or_fairness_risk,validation
0,f0,pre-treatment covariate,Criteo uplift source column f0,randomized unit,No invented semantic transformation; numeric v...,Measured before randomized assignment,yes,none,native numeric; source audit reports missingness,T-learner outcome model,NaN,low: pre-treatment source covariate,anonymous meaning prevents subgroup fairness i...,arm balance SMD and holdout-only uplift evalua...
1,f1,pre-treatment covariate,Criteo uplift source column f1,randomized unit,No invented semantic transformation; numeric v...,Measured before randomized assignment,yes,none,native numeric; source audit reports missingness,T-learner outcome model,NaN,low: pre-treatment source covariate,anonymous meaning prevents subgroup fairness i...,arm balance SMD and holdout-only uplift evalua...
2,f2,pre-treatment covariate,Criteo uplift source column f2,randomized unit,No invented semantic transformation; numeric v...,Measured before randomized assignment,yes,none,native numeric; source audit reports missingness,T-learner outcome model,NaN,low: pre-treatment source covariate,anonymous meaning prevents subgroup fairness i...,arm balance SMD and holdout-only uplift evalua...
3,f3,pre-treatment covariate,Criteo uplift source column f3,randomized unit,No invented semantic transformation; numeric v...,Measured before randomized assignment,yes,none,native numeric; source audit reports missingness,T-learner outcome model,NaN,low: pre-treatment source covariate,anonymous meaning prevents subgroup fairness i...,arm balance SMD and holdout-only uplift evalua...
4,f4,pre-treatment covariate,Criteo uplift source column f4,randomized unit,No invented semantic transformation; numeric v...,Measured before randomized assignment,yes,none,native numeric; source audit reports missingness,T-learner outcome model,NaN,low: pre-treatment source covariate,anonymous meaning prevents subgroup fairness i...,arm balance SMD and holdout-only uplift evalua...
5,f5,pre-treatment covariate,Criteo uplift source column f5,randomized unit,No invented semantic transformation; numeric v...,Measured before randomized assignment,yes,none,native numeric; source audit reports missingness,T-learner outcome model,NaN,low: pre-treatment source covariate,anonymous meaning prevents subgroup fairness i...,arm balance SMD and holdout-only uplift evalua...
6,f6,pre-treatment covariate,Criteo uplift source column f6,randomized unit,No invented semantic transformation; numeric v...,Measured before randomized assignment,yes,none,native numeric; source audit reports missingness,T-learner outcome model,NaN,low: pre-treatment source covariate,anonymous meaning prevents subgroup fairness i...,arm balance SMD and holdout-only uplift evalua...
7,f7,pre-treatment covariate,Criteo uplift source column f7,randomized unit,No invented semantic transformation; numeric v...,Measured before randomized assignment,yes,none,native numeric; source audit reports missingness,T-learner outcome model,NaN,low: pre-treatment source covariate,anonymous meaning prevents subgroup fairness i...,arm balance SMD and holdout-only uplift evalua...
8,f8,pre-treatment covariate,Criteo uplift source column f8,randomized unit,No invented semantic transformation; numeric v...,Measured before randomized assignment,yes,none,native numeric; source audit reports missingness,T-learner outcome model,NaN,low: pre-treatment source covariate,anonymous meaning prevents subgroup fairness i...,arm balance SMD and holdout-only uplift evalua...
9,f9,pre-treatment covariate,Criteo uplift source column f9,randomized unit,No invented semantic transformation; numeric v...,Measured before randomized assignment,yes,none,native numeric; source audit reports missingness,T-learner outcome model,NaN,low: pre-treatment source covariate,anonymous meaning prevents subg

,feature_family,model_role,features
0,model output,evaluation/ranking only,1
1,pre-treatment covariate,T-learner outcome model,12


## Decision-time audit

A feature is eligible only when its information exists at the stated observation cutoff. Model-fitted bins, encoders, imputers, and thresholds use development data only. Outcomes and evaluation groups never enter the predictor matrix.

In [3]:
audit = features[['feature_name', 'observation_cutoff', 'available_at_decision', 'fit_partition', 'leakage_risk', 'validation']]
display(audit)


,feature_name,observation_cutoff,available_at_decision,fit_partition,leakage_risk,validation
0,f0,Measured before randomized assignment,yes,none,low: pre-treatment source covariate,arm balance SMD and holdout-only uplift evalua...
1,f1,Measured before randomized assignment,yes,none,low: pre-treatment source covariate,arm balance SMD and holdout-only uplift evalua...
2,f2,Measured before randomized assignment,yes,none,low: pre-treatment source covariate,arm balance SMD and holdout-only uplift evalua...
3,f3,Measured before randomized assignment,yes,none,low: pre-treatment source covariate,arm balance SMD and holdout-only uplift evalua...
4,f4,Measured before randomized assignment,yes,none,low: pre-treatment source covariate,arm balance SMD and holdout-only uplift evalua...
5,f5,Measured before randomized assignment,yes,none,low: pre-treatment source covariate,arm balance SMD and holdout-only uplift evalua...
6,f6,Measured before randomized assignment,yes,none,low: pre-treatment source covariate,arm balance SMD and holdout-only uplift evalua...
7,f7,Measured before randomized assignment,yes,none,low: pre-treatment source covariate,arm balance SMD and holdout-only uplift evalua...
8,f8,Measured before randomized assignment,yes,none,low: pre-treatment source covariate,arm balance SMD and holdout-only uplift evalua...
9,f9,Measured before randomized assignment,yes,none,low: pre-treatment source covariate,arm balance SMD and holdout-only uplift evalua...


## Committed analytical evidence

In [4]:
evidence_files = ["reports/generated/ate_results.json", "reports/generated/uplift_policy_results.json"]
for relative in evidence_files:
    path = ROOT / relative
    display(Markdown(f'### `{relative}`'))
    if not path.exists():
        display(Markdown('*Not committed; reproduce locally using the README pipeline.*'))
    elif path.suffix == '.csv':
        display(pd.read_csv(path).head(20))
    elif path.suffix == '.json':
        payload = json.loads(path.read_text())
        display(payload)


### `reports/generated/ate_results.json`

{'conversion': {'absolute_effect': 0.0011518730521316279,
  'ci_high': 0.0012192415519387749,
  'ci_low': 0.0010845045523244809,
  'control_rate': 0.0019375880152813366,
  'incremental_outcomes_per_100k_assigned': 115.18730521316279,
  'mde_80_power_absolute': 9.228077217163973e-05,
  'number_needed_to_assign': 868.151223912587,
  'p_value': 3.194105972880377e-246,
  'relative_effect': 0.594488117725262,
  'standard_error': 3.437168357507502e-05,
  'treatment_rate': 0.0030894610674129645},
 'exposure_first_stage': 0.036036727482199896,
 'interpretation': {'itt': 'Primary causal estimate: effect of assignment, preserving randomization.',
  'iv': 'Wald scaling is descriptive and requires exclusion, monotonicity, and instrument relevance assumptions.',
  'per_protocol': 'Not reported as causal because observed exposure is post-treatment selection.'},
 'iv_complier_effect': {'conversion': 0.031963863885825596,
  'visit': 0.28699618005843747},
 'visit': {'absolute_effect': 0.010342403129198

### `reports/generated/uplift_policy_results.json`

{'deciles': [{'decile': 10,
   'ipw_observed_effect': 0.08533986928104578,
   'mean_predicted_uplift': 0.07129749530606136,
   'n': 300000,
   'standard_error': 0.0024104941055713707},
  {'decile': 9,
   'ipw_observed_effect': 0.009661437908496731,
   'mean_predicted_uplift': 0.009990148586233767,
   'n': 300000,
   'standard_error': 0.0012760433788599399},
  {'decile': 8,
   'ipw_observed_effect': 0.0014091503267973878,
   'mean_predicted_uplift': 0.002298876620753149,
   'n': 300000,
   'standard_error': 0.0006447386106776753},
  {'decile': 7,
   'ipw_observed_effect': 0.00040522875816993595,
   'mean_predicted_uplift': 0.0008709424395512035,
   'n': 300000,
   'standard_error': 0.00040313606109363727},
  {'decile': 6,
   'ipw_observed_effect': 0.0007228758169934639,
   'mean_predicted_uplift': 0.0004621427044686711,
   'n': 300000,
   'standard_error': 0.00023967630461881135},
  {'decile': 5,
   'ipw_observed_effect': 0.0004718954248366013,
   'mean_predicted_uplift': 0.000315520763

## Decision

**Average treatment assignment effect is positive, but the DR-versus-ITT gap and bottom-decile sign mismatch block targeted deployment.**

### Claim boundary

The notebook distinguishes observed evidence from model output, simulated policy, and causal impact. A strong offline metric is not production exposure.